In [ ]:
from platform import python_version
print(python_version())

In [ ]:
!echo $CONDA_DEFAULT_ENV

### Install Cellpose-SAM

https://pypi.org/project/cellpose/

https://github.com/MouseLand/cellpose/blob/main/notebooks/run_Cellpose-SAM.ipynb

In [ ]:
!nvidia-smi

In [ ]:
!nvcc --version

In [ ]:
import os, sys
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import trange
import matplotlib.pyplot as plt
from natsort import natsorted

In [ ]:
from cellpose import models, core, io, plot

#### run this to get printing of progress

  - cellpose version: 	4.0.7 
  - platform:       	linux 
  - python version: 	3.11.14 
  - torch version:  	2.9.0+cu128

In [ ]:
io.logger_setup()

#### Check if colab notebook instance has GPU access

In [ ]:
if core.use_gpu()==False:
    print(ImportError("No GPU access, change your runtime"))
else:
    print("GPU Ok")

### Select an image

In [ ]:
is_laptop = False

if is_laptop:
    root0 = "../../colaboracoes/carlos_deOcesano"
else:
    root0 = "/media/flalix/d2f268d1-512d-499f-b3b3-6dad7d3fdd25/colaboracoes/deOcesano"
    
os.listdir(root0)

In [ ]:
plate = 'Plate1848'
root_hcs = os.path.join(root0, plate)
hcs_folders = os.listdir(root_hcs)
print(root_hcs)
hcs_folders

In [ ]:
experiment='1% SFB'

root_img = os.path.join(root_hcs, experiment)
os.path.exists(root_img), root_img

In [ ]:
files = os.listdir(root_img)
print(">>>", len(files))
for fname in files[:5]:
    print(fname)

In [ ]:
i=15
fname=files[i]
filefig = os.path.join(root_img, fname)
img = io.imread(filefig)

fig = plt.figure(figsize=(8, 8))
plt.imshow(img);

In [ ]:
print(f"\nImage '{fname}' in '{root_img}' has shape: {img.shape}. Assuming channel dimension is last with {img.shape[-1]} channels\n\n")

### Crop an image

box = (left, upper, right, lower)

In [ ]:
img.shape

In [ ]:
from PIL import Image

image = Image.open(filefig)

In [ ]:
box = (0, 0, 248, 248)

In [ ]:
#box = (left, upper, right, lower)
seg_img = image.crop(box)

fig = plt.figure(figsize=(4, 4))
plt.imshow(seg_img);

### Cut 25 crops: 5 x 5

In [ ]:
width, height = image.size
ncrop = 5

del_width = int(width/ncrop)
del_height = int(height/ncrop)

del_width, del_height

In [ ]:
np.zeros( (5,5) )

### Reading the image

In [ ]:
from PIL import Image

image = Image.open(filefig)

In [ ]:
see_whole_image = False

if see_whole_image:
    fig = plt.figure(figsize=(8, 8))
    plt.imshow(image);

In [ ]:
dic = {}

fig, axes = plt.subplots(ncrop, ncrop, figsize=(8,8))

icount = -1; dic={}
for ix in range(ncrop):
    xmin = ix*del_width

    if ix == ncrop-1:
        # dont loose any bite
        xmax = width
    else:
        xmax = xmin + del_width
    
    for iy in range(ncrop):
        ymin = iy*del_height
        
        if iy == ncrop-1:
            # dont loose any bite
            ymax = height
        else:
            ymax = ymin + del_height

        # box = (left, upper, right, lower)
        box = [xmin, ymin, xmax, ymax]
        seg_img = image.crop(box)

        icount += 1
        dic[icount] = {}
        dic2 = dic[icount]

        dic2['xmin'] = xmin
        dic2['xmax'] = xmax
        dic2['ymin'] = ymin
        dic2['ymax'] = ymax

        dic2['del_x'] = xmax-xmin
        dic2['del_y'] = ymax-ymin
        
        dic2['image'] = seg_img

        ax = axes[iy][ix]
        ax.imshow(seg_img)
        ax.xaxis.set_visible(False)
        ax.yaxis.set_visible(False)

plt.subplots_adjust(left=0.1, bottom=0.1, right=0.9, top=0.9, wspace=0.05, hspace=0.05)
plt.tight_layout()

In [ ]:
dic.keys()

In [ ]:
df = pd.DataFrame(dic).T
df

### In parallel we save the table in 'tables'

In [ ]:
root_tables = os.path.join(root0, 'tables')
if not os.path.exists(root_tables):
    os.mkdir(root_tables)

In [ ]:
root_tsv = os.path.join(root_tables, plate)
if not os.path.exists(root_tsv):
    os.mkdir(root_tsv)

In [ ]:
fname

In [ ]:
fname_tsv = fname.replace('.tif', '.tsv')

filename = os.path.join(root_tsv, fname_tsv)

try:
    df.to_csv(filename, sep='\t')
except:
    print(f"Error: saving {filename}")

In [ ]:
n, mu, std = len(df), df.del_x.mean(), df.del_x.std()
f"The plate {plate} has {n} images: mean {mu:.2f} (std={std:.3f})"